# Verify the grouped VQ-VAE

In [ ]:
import json
import pathlib
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from exp_run_config import Config
from sensorprocessing.conv_vae_neo import DemonstrationImageDataset
from sensorprocessing.sp_factory import create_sp

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
experiment = "sensorprocessing_vq_vae"
run = "sp_vq_vae_128_256px"

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
sp = create_sp(exp)
dataset = DemonstrationImageDataset(exp, "validation_data")
loader = DataLoader(dataset, batch_size=exp["batch_size"], shuffle=False)

## Reconstruction error and code usage on the validation demonstrations

In [ ]:
counts = torch.zeros(exp["num_codes_per_image"], exp["codebook_size"], dtype=torch.long)
squared_error = 0.0
pixels = 0
with torch.no_grad():
    for images in loader:
        images = images.to(Config().runtime["device"])
        reconstruction, _, _, ids = sp.enc(images)
        squared_error += (reconstruction - images).square().sum().item()
        pixels += images.numel()
        for group, group_ids in enumerate(ids.cpu().T):
            counts[group] += torch.bincount(group_ids, minlength=exp["codebook_size"])

metrics = {
    "images": len(dataset),
    "reconstruction_mse": squared_error / pixels,
    "code_counts_per_group": counts.tolist(),
}
path = pathlib.Path(exp.data_dir(), "verification_metrics.json")
path.write_text(json.dumps(metrics, indent=2) + "\n")
print(f"Validation MSE: {metrics['reconstruction_mse']:.6f}")
print("Active codes per group:", counts.count_nonzero(dim=1).tolist())

In [ ]:
exp.done()